# VEILINK 六轴机械臂
# 二、舵机零位与限位校准

本 Notebook 用于已完成 ID 编号和机械装配后的整机校准。校准结果保存为同目录下的 `机械臂校准参数.json`，后续运动学、关节控制和手柄程序均应从该文件加载参数。

校准内容：

- 在设计零位记录 J1～J6 和夹爪的编码器零位；
- J1、J4、J6 按单圈 360°关节处理，原始范围固定为 0～4095；
- J2、J3、J5 和夹爪在关闭扭矩后手动采集机械安全范围；
- 生成 LeRobot 可读取的校准字段，并可选地把安全限位写入舵机 EEPROM。

> 校准文件中的 `calibrated` 只有在零位和所有必需限位均完成后才会变为 `true`。后续控制程序应拒绝使用 `calibrated=false` 的文件使能扭矩。

## 0. 安全要求

1. 确认七台舵机已经分别设置为 J1～J6/ID1～ID6、夹爪/ID7；不使用电动夹爪时可关闭夹爪选项。
2. 核对 STS3215 的具体电压版本、电源极性和电源容量。
3. 校准全过程默认关闭扭矩；肩部和肘部必须用支架、吊带或另一人托住，防止机械臂突然下坠。
4. 手动测量限位时，不要硬压结构止挡。将“首次接近干涉、线缆拉紧或阻力明显增加的位置”作为测量端点，程序还会向内增加安全裕量。
5. 手不要放在连杆、齿轮、夹爪之间的夹伤区域。出现发热、异味、异常响声或通信异常时立即断电。
6. J1、J4、J6 虽可整圈旋转，也必须确认外部线缆不会缠绕；整圈能力不等于允许无限多圈连续旋转。
7. 本 Notebook 不会主动转动关节。只有最后的可选步骤会把限位写入舵机 EEPROM。

## 1. 环境、路径与参数文件检查

选择 `Python (lerobot)` kernel 后运行下一格。程序会自动寻找 Notebook 同目录下的 `机械臂校准参数.json`。

In [ ]:
import json
import sys
import time
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

from IPython.display import clear_output
from serial.tools import list_ports

WORK_DIR = Path.cwd()
if WORK_DIR.name == "Code":
    NOTEBOOK_DIR = Path(".")
elif (WORK_DIR / "Code").is_dir():
    NOTEBOOK_DIR = Path("Code")
else:
    NOTEBOOK_DIR = Path(".")

PARAM_PATH = NOTEBOOK_DIR / "机械臂校准参数.json"
if not PARAM_PATH.exists():
    raise FileNotFoundError(f"找不到参数模板：{PARAM_PATH}")

calibration_state = json.loads(PARAM_PATH.read_text(encoding="utf-8"))
print("Python:", Path(sys.executable).name)
print("LeRobot version:", version("lerobot"))
print("参数文件:", PARAM_PATH)
print("当前 calibrated =", calibration_state["calibrated"])


## 1A. 同步最终运动学参数

三张最终图片已经确认：DH 表定义数值，运动学模型定义坐标轴与电机正转，零位图定义实体装配姿态。

$$
d=[-64,0,0,-171,0,-159]\ \mathrm{mm},\quad
a=[0,120,0,0,0,0]\ \mathrm{mm},
$$

$$
\alpha=[-90,180,-90,90,-90,0]^\circ,\quad
\theta_0=[0,90,90,180,0,0]^\circ.
$$

下面的单元把这些最终值同步到内存中的校准数据，同时加入设计范围和零位 TCP 验证值。它不会访问串口，也不会改变舵机。

In [ ]:
KINEMATICS_REVISION = "2026-08-09-dh-v2"
FINAL_DH_PARAMETERS = [
    {"joint":"J1", "function":"基座回转", "theta_expression":"q1",        "d_mm":-64,  "a_mm":0,   "alpha_deg":-90, "theta_offset_deg":0,   "design_min_deg":-180, "design_max_deg":180},
    {"joint":"J2", "function":"肩部俯仰", "theta_expression":"q2 + 90°",  "d_mm":0,    "a_mm":120, "alpha_deg":180, "theta_offset_deg":90,  "design_min_deg":-87,  "design_max_deg":87},
    {"joint":"J3", "function":"肘部俯仰", "theta_expression":"q3 + 90°",  "d_mm":0,    "a_mm":0,   "alpha_deg":-90, "theta_offset_deg":90,  "design_min_deg":-155, "design_max_deg":155},
    {"joint":"J4", "function":"前臂回转", "theta_expression":"q4 + 180°", "d_mm":-171, "a_mm":0,   "alpha_deg":90,  "theta_offset_deg":180, "design_min_deg":-180, "design_max_deg":180},
    {"joint":"J5", "function":"腕部俯仰", "theta_expression":"q5",        "d_mm":0,    "a_mm":0,   "alpha_deg":-90, "theta_offset_deg":0,   "design_min_deg":-101, "design_max_deg":101},
    {"joint":"J6", "function":"末端回转", "theta_expression":"q6",        "d_mm":-159, "a_mm":0,   "alpha_deg":0,   "theta_offset_deg":0,   "design_min_deg":0,    "design_max_deg":360},
]

def synchronize_final_kinematics(state):
    """只同步最终运动学元数据；不覆盖已经实测的零位和软限位。"""
    state["kinematics_status"] = "final_confirmed"
    state["kinematics_revision"] = KINEMATICS_REVISION
    state["kinematics_confirmed_at"] = "2026-08-09"
    state["kinematics_sources"] = [
        "运动学参数 Kinematic Parameters/运动学模型 Kinematic Model.png",
        "运动学参数 Kinematic Parameters/DH参数表 DH Parameters.png",
        "运动学参数 Kinematic Parameters/零位 Zero Pose.PNG",
    ]
    rows = []
    for source in FINAL_DH_PARAMETERS:
        row = dict(source)
        row["a_prev_mm"] = row["a_mm"]
        row["alpha_prev_deg"] = row["alpha_deg"]
        row["dh_theta_offset_deg"] = row["theta_offset_deg"]
        rows.append(row)
        joint = state["joints"][row["joint"]]
        joint["dh_theta_offset_deg"] = row["theta_offset_deg"]
        joint["design_min_deg"] = row["design_min_deg"]
        joint["design_max_deg"] = row["design_max_deg"]
        joint["positive_axis_reference"] = "最终运动学模型中的红色+z_(i-1)箭头"
        joint["direction"] = 1
        joint["direction_verified"] = True
        joint["direction_verification_source"] = "红色+z_(i-1)箭头按右手定则的正转就是舵机正方向"
        joint["direction_verified_at"] = "2026-08-09"

    gripper = state["joints"]["gripper"]
    gripper["direction"] = 1
    gripper["direction_verified"] = True
    gripper["direction_verification_source"] = '用户确认：夹爪采用相同舵机，最终运动学图片中的红色箭头按右手定则表示夹爪舵机正方向'
    gripper["direction_verified_at"] = '2026-08-09'
    state["direction_status"] = {
        "arm_joints_verified": True,
        "arm_joints": ["J1", "J2", "J3", "J4", "J5", "J6"],
        "arm_direction": 1,
        "source": "最终运动学模型红色箭头；正转按右手定则",
        "gripper_verified": True,
        "gripper_direction": 1,
        "gripper_source": "最终运动学模型的夹爪红色旋转箭头",
    }
    state["dh_parameters"] = rows
    state["coordinate_convention"].update({
        "dh_convention": "standard_DH",
        "dh_transform": "T(i-1,i)=Rz(theta_i)Tz(d_i)Tx(a_i)Rx(alpha_i)",
        "base_frame_origin": "ground center on the J1 rotation axis",
        "base_z_positive": "toward ground",
        "zero_pose_q_deg": [0, 0, 0, 0, 0, 0],
        "zero_pose_tcp_position_mm": [0, 0, -514],
        "zero_pose_tcp_rotation_matrix": [[-1,0,0],[0,-1,0],[0,0,1]],
        "servo_positive_direction": "J1-J6红色+z_(i-1)箭头按右手定则的正转就是舵机正方向",
        "note": "已按 2026-08-09-dh-v2 同步标准DH参数；J1-J6电机正转以模型红色箭头为准。",
    })
    return state

synchronize_final_kinematics(calibration_state)
print("最终运动学参数已同步到内存；后续保存单元会写回 JSON。")
for row in calibration_state["dh_parameters"]:
    print(row["joint"], row["theta_expression"], "d=", row["d_mm"], "a=", row["a_mm"], "alpha=", row["alpha_deg"])

## 2. 查找串口并填写硬件配置

先运行串口扫描，再把 `PORT` 改成控制板的实际端口。若夹爪未安装或不是 STS3215，将 `INCLUDE_GRIPPER=False`。

In [ ]:
ports = list(list_ports.comports())
if not ports:
    print("未检测到串口，请检查 USB 线、驱动和控制板。")
else:
    for index, port in enumerate(ports, start=1):
        print(f"{index}. {port.device:8s} | {port.description} | {port.hwid}")

# TODO：改成实际串口。
PORT = "COM3"
INCLUDE_GRIPPER = True

JOINT_ORDER = ["J1", "J2", "J3", "J4", "J5", "J6"]
if INCLUDE_GRIPPER:
    JOINT_ORDER.append("gripper")

MANUAL_LIMIT_JOINTS = [joint for joint in ["J2", "J3", "J5", "gripper"] if joint in JOINT_ORDER]
CONTINUOUS_JOINTS = ["J1", "J4", "J6"]

calibration_state["hardware"]["port_used"] = PORT
calibration_state["hardware"]["gripper_enabled"] = INCLUDE_GRIPPER
calibration_state["lerobot_version"] = version("lerobot")

print("PORT =", PORT)
print("参与校准的舵机：", JOINT_ORDER)
print("需要手动采集限位：", MANUAL_LIMIT_JOINTS)


## 3. 创建总线、连接整机并关闭扭矩

运行后会校验所有规划 ID，并立即关闭全部舵机扭矩。不会发送目标位置。若某个 ID 掉线，应先断电检查编号、接线和供电。

In [ ]:
from lerobot.motors import Motor, MotorCalibration, MotorNormMode
from lerobot.motors.feetech import FeetechMotorsBus, OperatingMode

ENCODER_RESOLUTION = 4096
ENCODER_MAX = ENCODER_RESOLUTION - 1

def make_bus():
    motors = {}
    for joint in JOINT_ORDER:
        item = calibration_state["joints"][joint]
        mode = MotorNormMode.RANGE_0_100 if joint == "gripper" else MotorNormMode.DEGREES
        motors[joint] = Motor(item["id"], "sts3215", mode)
    return FeetechMotorsBus(port=PORT, motors=motors)

bus = make_bus()
bus.connect(handshake=True)
bus.disable_torque(num_retry=2)
for joint in JOINT_ORDER:
    current_mode = int(bus.read("Operating_Mode", joint, normalize=False, num_retry=2))
    if current_mode != OperatingMode.POSITION.value:
        print(f"{joint} 当前工作模式为 {current_mode}，改为位置模式。")
        bus.write("Operating_Mode", joint, OperatingMode.POSITION.value, normalize=False, num_retry=2)

positions = bus.sync_read("Present_Position", normalize=False, num_retry=2)
voltages = bus.sync_read("Present_Voltage", normalize=False, num_retry=2)
temperatures = bus.sync_read("Present_Temperature", normalize=False, num_retry=2)
print("总线连接成功，全部舵机扭矩已关闭：")
for joint in JOINT_ORDER:
    print(f"{joint:8s} ID={calibration_state['joints'][joint]['id']} pos={positions[joint]} voltage_raw={voltages[joint]} temp_raw={temperatures[joint]}")


## 4. 保存与角度换算辅助函数

保存采用临时文件替换，避免 Notebook 中断时留下半个 JSON。`wrapped_tick_delta` 用于处理 J1/J4/J6 越过 0/4095 时的单圈角度差。

In [ ]:
def save_calibration_state():
    synchronize_final_kinematics(calibration_state)
    temp_path = PARAM_PATH.with_suffix(".json.tmp")
    temp_path.write_text(
        json.dumps(calibration_state, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temp_path.replace(PARAM_PATH)
    print("已保存：", PARAM_PATH)

def wrapped_tick_delta(value, reference):
    return ((int(value) - int(reference) + ENCODER_RESOLUTION // 2) % ENCODER_RESOLUTION) - ENCODER_RESOLUTION // 2

def raw_to_joint_degrees(joint, raw_value):
    item = calibration_state["joints"][joint]
    if item["zero_position_raw"] is None:
        raise RuntimeError(f"{joint} 尚未记录零位。")
    delta = wrapped_tick_delta(raw_value, item["zero_position_raw"])
    return item["direction"] * delta * 360.0 / ENCODER_RESOLUTION

save_calibration_state()


## 5. 设置机械/DH 零位

将机械臂摆到最终运动学图片定义的 $q=[0,0,0,0,0,0]^T$ 零位：主轴竖直，J1/J4/J6 按红色正轴参考对齐，夹爪竖直。该零位的理论 TCP 为 $[0,0,-514]^T$ mm，DH 姿态矩阵为 $\operatorname{diag}(-1,-1,1)$。
必须先托住机械臂，确认姿态准确后，再运行下面的代码。

In [ ]:
if not bus.is_connected:
    raise RuntimeError("总线未连接，请重新运行第 3 节。")
bus.disable_torque(num_retry=2)
input("最后检查零位姿态。确认无误后按 Enter 写入 Homing Offset。")
homing_offsets = bus.set_half_turn_homings(JOINT_ORDER)
zero_positions = bus.sync_read("Present_Position", JOINT_ORDER, normalize=False, num_retry=2)

for joint in JOINT_ORDER:
    item = calibration_state["joints"][joint]
    item["homing_offset"] = int(homing_offsets[joint])
    item["zero_position_raw"] = int(zero_positions[joint])
    if joint in CONTINUOUS_JOINTS:
        item["measured_min_raw"] = 0
        item["measured_max_raw"] = ENCODER_MAX
        item["soft_min_raw"] = 0
        item["soft_max_raw"] = ENCODER_MAX

calibration_state["calibrated"] = False
calibration_state["servo_eeprom_limits_written"] = False
save_calibration_state()
print("零位记录完成：")
for joint in JOINT_ORDER:
    print(f"{joint:8s} zero_raw={zero_positions[joint]} homing_offset={homing_offsets[joint]}")


## 6. 手动采集 J2、J3、J5 和夹爪限位

本节会在同一个记录窗口内，同时采集 J2、J3、J5 和夹爪。你可以在倒计时内依次活动它们，但程序会持续同步读取全部待校准关节。

运行流程：

1. 程序先关闭全部舵机扭矩；
2. 读取 J1、J4、J6 的当前位置，并先把当前位置写为目标位置；
3. 只给 J1、J4、J6 使能扭矩，使其保持当前位置，防止随意转动；
4. J2、J3、J5 和夹爪保持无扭矩，在同一记录窗口内手动移动到各自两个安全端点；
5. 程序持续监视 J1/J4/J6 的保持误差；误差过大将中止采集；
6. 无论采集成功、报错或手动中断，程序都会在退出时关闭 J1/J4/J6 的保持扭矩；
7. 完成后统一计算并保存所有关节限位。J2/J3/J5 默认向内保留 5°，夹爪保留 2°。

> J1/J4/J6 使能保持扭矩后可能有很小的定位动作。J2/J3/J5/J7 仍然没有扭矩，必须托住机械臂。

In [ ]:
HOLD_JOINTS = [joint for joint in ["J1", "J4", "J6"] if joint in JOINT_ORDER]
HOLD_MAX_ERROR_TICKS = 120  # 约 10.5°；超过时立即中止并关闭保持扭矩。

def record_manual_ranges(joints, hold_joints, seconds=60):
    if not joints:
        raise ValueError("没有需要手动采集限位的关节。")
    for joint in joints:
        if joint not in MANUAL_LIMIT_JOINTS:
            raise ValueError(f"{joint} 不属于手动限位关节：{MANUAL_LIMIT_JOINTS}")
        if calibration_state["joints"][joint]["zero_position_raw"] is None:
            raise RuntimeError(f"{joint} 尚未记录零位，请先完成第 5 节。")

    # 先让全部关节失能，再把保持关节的“当前位置”写为目标位置。
    bus.disable_torque(num_retry=2)
    hold_targets = {
        joint: int(value)
        for joint, value in bus.sync_read("Present_Position", hold_joints, normalize=False, num_retry=2).items()
    }
    bus.sync_write("Goal_Position", hold_targets, normalize=False, num_retry=2)

    start_positions = {
        joint: int(value)
        for joint, value in bus.sync_read("Present_Position", joints, normalize=False, num_retry=2).items()
    }
    observed_mins = start_positions.copy()
    observed_maxes = start_positions.copy()
    previous_positions = start_positions.copy()
    wrap_detected = {joint: False for joint in joints}
    last_display = 0.0

    print("保持目标：", hold_targets)
    print(f"即将给 {hold_joints} 使能保持扭矩。")
    time.sleep(0.5)

    try:
        bus.enable_torque(hold_joints, num_retry=2)
        deadline = time.time() + seconds
        while time.time() < deadline:
            positions = {
                joint: int(value)
                for joint, value in bus.sync_read("Present_Position", joints, normalize=False, num_retry=2).items()
            }
            for joint in joints:
                # 手动运动不可能在一个采样周期内跨越半圈；若原始值突然大跳，
                # 才说明真实经过了 4095 -> 0 或 0 -> 4095 回绕点。
                if abs(positions[joint] - previous_positions[joint]) > ENCODER_RESOLUTION // 2:
                    wrap_detected[joint] = True
                observed_mins[joint] = min(observed_mins[joint], positions[joint])
                observed_maxes[joint] = max(observed_maxes[joint], positions[joint])
                previous_positions[joint] = positions[joint]

            held_positions = {
                joint: int(value)
                for joint, value in bus.sync_read("Present_Position", hold_joints, normalize=False, num_retry=2).items()
            }
            hold_errors = {
                joint: wrapped_tick_delta(held_positions[joint], hold_targets[joint])
                for joint in hold_joints
            }
            excessive = {joint: error for joint, error in hold_errors.items() if abs(error) > HOLD_MAX_ERROR_TICKS}
            if excessive:
                raise RuntimeError(f"保持关节位置误差过大：{excessive}")

            now = time.time()
            if now - last_display >= 0.5:
                clear_output(wait=True)
                print(f"正在同时记录 {joints}，剩余 {max(0, deadline-now):.1f} 秒")
                print(f"{'JOINT':<9} | {'MIN':>6} | {'POS':>6} | {'MAX':>6}")
                for joint in joints:
                    print(f"{joint:<9} | {observed_mins[joint]:>6} | {positions[joint]:>6} | {observed_maxes[joint]:>6}")
                print("保持关节误差(ticks)：", hold_errors)
                last_display = now
            time.sleep(0.03)
    finally:
        # 成功、异常或用户中断时都必须释放保持关节。
        bus.disable_torque(hold_joints, num_retry=2)
        print(f"已关闭 {hold_joints} 的保持扭矩。")

    # 先验证全部结果，再统一写入内存，避免部分关节成功、部分关节失败。
    results = {}
    for joint in joints:
        observed_min = observed_mins[joint]
        observed_max = observed_maxes[joint]
        span = observed_max - observed_min
        if span < 50:
            raise ValueError(f"{joint} 记录范围过小（{span} ticks），请重新同时采集全部限位。")
        if wrap_detected[joint]:
            raise ValueError(
                f"{joint} 在采集过程中实际跨越了 0/4095 回绕点。"
                "有限关节的舵机零位应尽量位于可用行程中间，请检查舵盘安装或重新设置零位。"
            )

        item = calibration_state["joints"][joint]
        margin_ticks = round(item["safety_margin_deg"] * ENCODER_RESOLUTION / 360.0)
        soft_min = observed_min + margin_ticks
        soft_max = observed_max - margin_ticks
        if soft_min >= soft_max:
            raise ValueError(f"{joint} 的安全裕量大于可用范围。")
        results[joint] = {
            "measured_min_raw": observed_min,
            "measured_max_raw": observed_max,
            "soft_min_raw": soft_min,
            "soft_max_raw": soft_max,
        }

    for joint, values in results.items():
        item = calibration_state["joints"][joint]
        item.update(values)
        if joint != "gripper":
            endpoint_angles = [raw_to_joint_degrees(joint, values["soft_min_raw"]), raw_to_joint_degrees(joint, values["soft_max_raw"])]
            item["soft_min_deg"] = round(min(endpoint_angles), 4)
            item["soft_max_deg"] = round(max(endpoint_angles), 4)

    calibration_state["calibrated"] = False
    calibration_state["servo_eeprom_limits_written"] = False
    save_calibration_state()
    return results

RECORD_SECONDS = 60


input(
    f"将同时记录 {MANUAL_LIMIT_JOINTS}；{HOLD_JOINTS} 将保持当前位置。"
    f"确认支撑可靠后按 Enter 开始 {RECORD_SECONDS} 秒记录。"
)
results = record_manual_ranges(MANUAL_LIMIT_JOINTS, HOLD_JOINTS, RECORD_SECONDS)
print("全部手动限位采集完成：")
for joint, values in results.items():
    print(f"{joint:8s} measured=({values['measured_min_raw']}, {values['measured_max_raw']}) soft=({values['soft_min_raw']}, {values['soft_max_raw']})")


## 7. 汇总、校验并完成参数文件

只有以下条件全部满足才会标记校准完成：

- 所有关节已经记录零位和 Homing Offset；
- J2/J3/J5/夹爪已经记录安全限位。

本节还会生成 `lerobot_calibration` 字段，供后续程序构造 `MotorCalibration`。

In [ ]:
missing = []
if calibration_state.get("kinematics_status") != "final_confirmed" or calibration_state.get("kinematics_revision") != KINEMATICS_REVISION:
    missing.append("最终运动学参数")
if any(row.get("theta_offset_deg") is None for row in calibration_state.get("dh_parameters", [])):
    missing.append("DH固定角偏置")
for joint in JOINT_ORDER:
    item = calibration_state["joints"][joint]
    if item["zero_position_raw"] is None or item["homing_offset"] is None:
        missing.append(f"{joint}:零位")
    if item["manual_limit_required"] and (item["soft_min_raw"] is None or item["soft_max_raw"] is None):
        missing.append(f"{joint}:限位")

lerobot_calibration = {}
if not missing:
    for joint in JOINT_ORDER:
        item = calibration_state["joints"][joint]
        lerobot_calibration[joint] = {
            "id": item["id"],
            "drive_mode": 0 if item["direction"] == 1 else 1,
            "homing_offset": item["homing_offset"],
            "range_min": item["soft_min_raw"],
            "range_max": item["soft_max_raw"],
        }
    calibration_state["lerobot_calibration"] = lerobot_calibration
    calibration_state["calibrated"] = True
    calibration_state["calibrated_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
else:
    calibration_state["calibrated"] = False

save_calibration_state()
print("calibrated =", calibration_state["calibrated"])
if missing:
    print("尚未完成：")
    for entry in missing:
        print(" -", entry)
else:
    print("校准参数完整，可以供后续程序加载。")

print("\n关节汇总：")
for joint in JOINT_ORDER:
    item = calibration_state["joints"][joint]
    print(f"{joint:8s} zero={item['zero_position_raw']} soft_raw=({item['soft_min_raw']}, {item['soft_max_raw']})")


## 8. 后续程序加载方式

后续程序应先加载 JSON，检查 `calibrated==true`，再创建总线。不要把零位、方向或限位硬编码到控制程序里。

J1/J4/J6 的当前单圈角度应使用参数文件中的公式和角度解包；若要连续累计多圈角度，后续实时控制程序还需根据相邻采样跨越 0/4095 的情况维护圈数。

In [ ]:
def load_arm_calibration(path=PARAM_PATH):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if data.get("schema_version") != 1:
        raise ValueError("不支持的校准文件版本。")
    if not data.get("calibrated"):
        raise RuntimeError("校准尚未完成，禁止用于带扭矩运动。")
    if data.get("kinematics_status") != "final_confirmed" or data.get("kinematics_revision") != KINEMATICS_REVISION:
        raise RuntimeError("运动学参数不是 2026-08-09-dh-v2，禁止用于控制。")
    if any(row.get("theta_offset_deg") is None for row in data.get("dh_parameters", [])):
        raise RuntimeError("DH固定角偏置不完整，禁止用于控制。")
    return data

print("后续加载示例：params = load_arm_calibration()")
if calibration_state["calibrated"]:
    params = load_arm_calibration()
    print("加载验证成功：", params["robot_name"], params["calibrated_at"])


## 9. 结束与故障排查

完成后运行最后一格关闭扭矩和串口，再断开舵机电源。

常见问题：

- **某个 ID 无响应**：断电后检查该舵机标签、串联线缆和供电压降，并用编号 Notebook 单独验证。
- **手动移动困难**：确认扭矩确实关闭；不要强行扳动，先断电排查机械干涉。
- **范围跨越 0/4095**：说明当前零位没有把该有限关节放在编码器中间附近，应重新检查舵盘安装或重新执行零位步骤。
- **零位记录错误**：重新摆到设计零位并再次执行第 5 节；这会使已有手动限位失效，必须重新采集。
- **`calibrated` 仍为 false**：运行第 7 节查看缺少的零位或限位项目。
- **DH 角度仍不正确**：最终固定偏置已经确认，依次为 `[0,-90,+90,+180,0,0]°`。若实机仍不一致，应检查编码器零位、舵盘安装和 `direction`，不要修改最终 DH 偏置来补偿装配误差。

In [ ]:
if "bus" in globals() and bus.is_connected:
    bus.disable_torque(num_retry=2)
    bus.disconnect(disable_torque=True)
    print("全部舵机扭矩已关闭，串口已断开。现在可以关闭舵机电源。")
else:
    print("总线当前未连接。")
